In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.datasets import cifar10
from keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# STEP 1: Load and Preprocess Data
# -------------------------------
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Flatten label arrays
y_train = y_train.flatten()
y_test = y_test.flatten()

# Select desired classes (0=airplane, 1=automobile, 2=bird, 3=cat)
selected_classes = [0, 1, 2, 3]
train_filter = np.isin(y_train, selected_classes)
test_filter = np.isin(y_test, selected_classes)

# Apply filters
X_train, y_train = X_train[train_filter], y_train[train_filter]
X_test, y_test = X_test[test_filter], y_test[test_filter]

# Normalize images
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Remap labels to 0–3
class_map = {c: i for i, c in enumerate(selected_classes)}
y_train = np.vectorize(class_map.get)(y_train)
y_test = np.vectorize(class_map.get)(y_test)

# One-hot encode
y_train = to_categorical(y_train, num_classes=4)
y_test = to_categorical(y_test, num_classes=4)

# -------------------------------
# STEP 2: Build CNN Model
# -------------------------------
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

# Display model summary
model.summary()

# -------------------------------
# STEP 3: Compile and Train
# -------------------------------
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=10,           # increased for better learning
    batch_size=64,
    validation_data=(X_test, y_test)
)

# -------------------------------
# STEP 4: Plot Accuracy and Loss
# -------------------------------
plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()
